# 01 — DCGAN @ 64px (baseline)This model exists to be beaten. Its jobs are to validate the loop end to end,give `docs/results.md` a reference point that makes FastGAN's improvementlegible, and demonstrate mode collapse first-hand on ~1,308 distinct shapes.Train it from a terminal — it checkpoints and resumes properly:```bashpython -m src.train_dcgan --name dcgan64 --steps 30000```

In [ ]:
import sys; sys.path.insert(0, "..")from pathlib import Pathimport torchimport matplotlib.pyplot as pltfrom tensorboard.backend.event_processing.event_accumulator import EventAccumulator

In [ ]:
def load_events(run):    ea = EventAccumulator(str(sorted(Path(f"../runs/{run}").glob("events.*"))[-1]))    ea.Reload()    return eadef plot_tags(ea, tags):    fig, axes = plt.subplots(1, len(tags), figsize=(4.7 * len(tags), 3.4))    for tag, ax in zip(tags, [axes] if len(tags) == 1 else axes):        pts = ea.Scalars(tag)        ax.plot([p.step for p in pts], [p.value for p in pts])        ax.set_title(tag); ax.set_xlabel("step")        ax.spines[["top", "right"]].set_visible(False)    plt.tight_layout(); plt.show()def plot_accuracy(ea):    """Both curves pinned at ~1.0 means D has memorised the training set."""    fig, ax = plt.subplots(figsize=(7, 3.2))    for tag in ("train/acc_real", "train/acc_fake"):        pts = ea.Scalars(tag)        ax.plot([p.step for p in pts], [p.value for p in pts], label=tag)    ax.axhline(1.0, ls="--", c="crimson", lw=1, label="memorisation")    ax.set_ylim(0, 1.05); ax.set_xlabel("step"); ax.legend()    ax.spines[["top", "right"]].set_visible(False)    plt.tight_layout(); plt.show()

## Architecture

In [ ]:
from src.config import DCGANConfigfrom src.models.dcgan import build_modelscfg = DCGANConfig()netG, netD = build_models(cfg.z_dim, cfg.g_channels, cfg.d_channels)print(f"G params: {sum(p.numel() for p in netG.parameters()) / 1e6:.1f}M")print(f"D params: {sum(p.numel() for p in netD.parameters()) / 1e6:.1f}M")print("output:", tuple(netG(torch.randn(2, cfg.z_dim)).shape))

## Loss curves

In [ ]:
ea = load_events("dcgan64")plot_tags(ea, ["train/d", "train/g"])

### The tell`acc_real` and `acc_fake` both pinned near 1.0 means D has won outright and Greceives no useful gradient. On ~1,308 shapes with no augmentation that is theexpected DCGAN failure, and it is the entire motivation for Phase 2'sDiffAugment and self-supervised discriminator.

In [ ]:
plot_accuracy(ea)

## Samples over training

In [ ]:
from PIL import Imageframes = sorted(Path("../samples/dcgan64").glob("*.png"))picks = [frames[0], frames[len(frames) // 2], frames[-1]] if len(frames) >= 3 else framesfig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 5.2))for ax, f in zip([axes] if len(picks) == 1 else axes, picks):    ax.imshow(Image.open(f)); ax.axis("off"); ax.set_title(f"step {int(f.stem)}")plt.tight_layout(); plt.show()